In [ ]:
import pandas as pd
import numpy as np
import json
import os
from copy import deepcopy,copy
from functools import reduce
from tqdm import tqdm

In [151]:
src_label=pd.read_excel("../../data/raw/※ 근육주사_행위 및 구두 단위 분석_Time Check_Final.xlsx", sheet_name="Data_상세시나리오")

with open("./sample.json", "r") as f:
    sample_label=json.load(f)
    
with open("./match_sample.json", "r") as f:
    match_sample_label=json.load(f)

In [ ]:
def extract_keys(json_obj, parent_key=""):
    keys = []
    if isinstance(json_obj, dict):
        for key, value in json_obj.items():
            new_key = f"{parent_key}_{key}" if parent_key else key
            keys.append(new_key)
            keys.extend(extract_keys(value, new_key))  # 재귀 호출
    elif isinstance(json_obj, list):
        for i, item in enumerate(json_obj):
            new_key = f"{parent_key}[{i}]"
            keys.append(new_key)
            keys.extend(extract_keys(item, new_key))  # 리스트 항목 탐색
    return keys

# 초기화
for i in tqdm(range(200)):
    index = "D" + str(i+1)
    file_index = "D" + str(i+1).zfill(3)

    dst_label = deepcopy(sample_label)  # 전체 딕셔너리 깊은 복사
    tree_list = extract_keys(dst_label)
    value = deepcopy(match_sample_label)  # 데이터 복사 (원본 변경 방지)

    for j in range(len(tree_list)):
        key_path = tree_list[j].split('_')  # 경로를 리스트로 변환
        
        # 마지막 키를 제외한 부모 객체 찾기
        parent_keys = key_path[:-1]
        last_key = key_path[-1]

        # parent_dict를 찾아서 최종 키를 접근할 준비
        parent_dict = reduce(lambda d, k: d[k], parent_keys, value) if parent_keys else value
        match_index = parent_dict[last_key]

        if isinstance(match_index, dict):  # 값이 딕셔너리라면 스킵
            continue
        
        match_index -= 1  # match_index를 0 기반 인덱스로 변경

        # dst_label에서도 같은 방식으로 접근
        parent_dict_dst = reduce(lambda d, k: d[k], parent_keys, dst_label) if parent_keys else dst_label
        dst_value = parent_dict_dst[last_key]  # 값만 얕은 복사

        # src_label 값에 따라 업데이트
        if src_label.loc[match_index][index] == 'O':
            dst_value = True
        elif src_label.loc[match_index][index] == 'X':
            dst_value = False
        elif src_label.loc[match_index][index] == 'M':
            dst_value = 'M'
        elif src_label.loc[match_index][index].find('X') != -1:
            if src_label.loc[match_index][index].find(last_key) != -1:
                dst_value = False
            else:
                dst_value = True

        # 변경된 값을 다시 저장
        parent_dict_dst[last_key] = dst_value  # 값만 수정
    

In [225]:
original = [[1, 2, 3], [4, 5, 6]]

# 얕은 복사 수행
shallow_copy = copy(original)

# 복사본 수정 (중첩 리스트 수정)
shallow_copy[0][0] = 100

print(original)       # [[100, 2, 3], [4, 5, 6]] (원본도 변경됨)
print(shallow_copy)   # [[100, 2, 3], [4, 5, 6]]

[[100, 2, 3], [4, 5, 6]]
[[100, 2, 3], [4, 5, 6]]
